# Notebook 03 - Segmentation Training Review

**Ramdas Chaugale - 40494109**  
**ECS8056 Themed Research Project**

Reviews the ten training runs carried out on the HPC five models at two input sizes from the
histories stored in their checkpoints. Nothing is retrained here.

Notebooks run in order: 01 → 02 → 03 → 04 → 05.

## 1. Setup

Paths resolve from the project root, so the notebook is location-independent. The ten checkpoints
are loaded once here and every later section reads from them.

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import torch

# this notebook runs from notebooks/, so the project root is one level up
PROJECT_ROOT = os.path.dirname(os.path.abspath(os.getcwd()))

CKPT_256    = os.path.join(PROJECT_ROOT, 'reconstruction', 'checkpoints_256')
CKPT_NATIVE = os.path.join(PROJECT_ROOT, 'reconstruction', 'checkpoints')

IMAGES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'images')
CURVES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'training_curves')
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(CURVES_DIR, exist_ok=True)

print(f"256 checkpoints:    {CKPT_256}")
print(f"Native checkpoints: {CKPT_NATIVE}")

In [ ]:
# one plotting style for the whole notebook, and one colour per model so the
# same model is the same colour in every figure below
plt.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 150, 'font.size': 11,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'legend.frameon': False,
    'figure.titlesize': 14, 'figure.titleweight': 'bold',
})

# fixed order, so every table and figure lists the models the same way
MODEL_FILES = {
    'Plain U-Net':         'plain_unet.pth',
    'Hybrid U-Net':        'hybrid_unet.pth',
    'OCT-F Backbone':      'octf_backbone.pth',
    'Modified OCT-F 2ch':  'modified_octf_2ch.pth',
    'Soft Erosion Hybrid': 'hybrid_soft_erosion.pth',
}

COLOURS = {
    'Plain U-Net':         '#2F6FA8',
    'Hybrid U-Net':        '#C0392B',
    'OCT-F Backbone':      '#2E9E5B',
    'Modified OCT-F 2ch':  '#E8A33D',
    'Soft Erosion Hybrid': '#7D5BA6',
}

print("Model order and colours fixed:")
for name in MODEL_FILES:
    print(f"  {name:<22} {COLOURS[name]}")

In [ ]:
# each checkpoint carries its weights, its full per-epoch history and the
# configuration it was trained under.
ckpt_256 = {}
ckpt_native = {}
for name in MODEL_FILES:
    path_256 = os.path.join(CKPT_256, MODEL_FILES[name])
    path_native = os.path.join(CKPT_NATIVE, MODEL_FILES[name])
    assert os.path.exists(path_256), f"MISSING: {path_256}"
    assert os.path.exists(path_native), f"MISSING: {path_native}"
    ckpt_256[name] = torch.load(path_256, map_location='cpu', weights_only=False)
    ckpt_native[name] = torch.load(path_native, map_location='cpu', weights_only=False)

# the input size is read out of the checkpoint rather than hardcoded, and the
# filename suffix keeps the two runs' figures apart
SIZE_256 = 'x'.join(map(str, ckpt_256['Plain U-Net']['config']['img_size']))
SIZE_NATIVE = 'x'.join(map(str, ckpt_native['Plain U-Net']['config']['img_size']))

print(f"Loaded {len(ckpt_256)} checkpoints at {SIZE_256} "
      f"and {len(ckpt_native)} at {SIZE_NATIVE}")

All ten checkpoints load. Because configuration and history travel with the weights, every
setting and every curve below is read back from the artefact that produced it rather than restated,
so this notebook cannot drift from the runs it describes.

## 2. Model specification and controlled differences

Parameter counts, input width, loss function and learning rates, read from the checkpoints. The
difference within each pair is what makes the comparison in notebook 04 interpretable.

In [ ]:
# the OCT-F models store two learning rates, one for the pretrained encoder and
# a larger one for the decoder that starts from random weights
print(f"{'Model':<22} {'Params':>10} {'In':>3} {'Loss':<18} {'Learning rate':<22}")
print("=" * 78)
for name in MODEL_FILES:
    config = ckpt_native[name]['config']
    rate = config['lr']
    if isinstance(rate, list):
        rate_text = f"enc {rate[0]:g} / dec {rate[1]:g}"
    else:
        rate_text = f"{rate:g}"
    print(f"{name:<22} {config['params']:>10,} {config['input_channels']:>3} "
          f"{config['loss_function']:<18} {rate_text:<22}")

In [ ]:
# within each pair only one thing changes, so any difference in the result
# comes from that variable rather than from model capacity
plain        = ckpt_native['Plain U-Net']['config']['params']
hybrid       = ckpt_native['Hybrid U-Net']['config']['params']
octf         = ckpt_native['OCT-F Backbone']['config']['params']
modified     = ckpt_native['Modified OCT-F 2ch']['config']['params']
soft_erosion = ckpt_native['Soft Erosion Hybrid']['config']['params']

print("Controlled pairs")
print(f"  Plain vs Hybrid U-Net:   {hybrid - plain:+,} params  (1ch to 2ch)")
print(f"  OCT-F vs Modified OCT-F: {modified - octf:+,} params  (1ch to 2ch)")
print(f"  Hybrid vs Soft Erosion:  {soft_erosion - hybrid:+,} params  (loss function only)")

Plain → Hybrid differs by **+288** parameters, which is a single 3x3 convolution widened from one
input channel to two. OCT-F → Modified OCT-F differs by **+384**. Hybrid → Soft Erosion differs by
**+0**  the loss function is the only change. Each pair therefore isolates one variable.

## 3. Protocol and convergence at 256 x 256

The settings shared by all five runs at this input size, then where each one stopped. The epoch
retained is the one with the lowest validation loss, not the last epoch trained.

In [ ]:
config = ckpt_256['Plain U-Net']['config']
print(f"Input size:      {SIZE_256}")
print(f"Batch size:      {config['batch_size']}")
print(f"Patience:        {config['patience']}")
print(f"PP kernel:       {config['pp_kernel']}")
print(f"Erosion kernel:  {config['erosion_kernel']}")
print("Split:           59 train / 15 validation, seed 40494109")

In [ ]:
# the epoch kept is the one with the lowest validation loss, not the last
print(f"{'Model':<22} {'Epochs':>7} {'Best':>6} {'Stop':>6} "
      f"{'Val loss':>10} {'Val dice':>10}")
print("=" * 64)
for name in MODEL_FILES:
    checkpoint = ckpt_256[name]
    n_epochs = len(checkpoint['history']['train_loss'])
    stop = checkpoint['config']['early_stopped_at']
    print(f"{name:<22} {n_epochs:>7} {checkpoint['epoch']:>6} {stop:>6} "
          f"{checkpoint['val_loss']:>10.4f} {checkpoint['val_dice']:>10.4f}")

All five terminated at `best + 10`, the early-stopping criterion, between epochs 47 and 66. None
reached the 150-epoch ceiling, so every run converged on its own terms. Validation loss at the
saved epoch spans 0.0656 to 0.1137.

## 4. Training curves at 256 x 256

Loss, Dice and learning rate per epoch for each model, then all five on shared axes. The dotted
line marks the epoch whose weights were kept.

In [ ]:
# one figure per model. the gap between the solid and dashed Dice lines is the
# generalisation signal, and the dotted line marks the epoch actually kept
for name in MODEL_FILES:
    checkpoint = ckpt_256[name]
    history = checkpoint['history']
    colour = COLOURS[name]
    best = checkpoint['epoch']
    stop = checkpoint['config']['early_stopped_at']
    epochs = np.arange(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

    axes[0].plot(epochs, history['train_loss'], color=colour, label='train')
    axes[0].plot(epochs, history['val_loss'], color=colour, linestyle='--',
                 label='validation')
    axes[0].axvline(best, color='0.4', linestyle=':', label=f"best {best}")
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('loss')
    axes[0].set_title('Loss')
    axes[0].legend(fontsize=9)

    axes[1].plot(epochs, history['train_dice'], color=colour, label='train')
    axes[1].plot(epochs, history['val_dice'], color=colour, linestyle='--',
                 label='validation')
    axes[1].axvline(best, color='0.4', linestyle=':')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('Dice')
    axes[1].set_title('Dice, higher is better')
    axes[1].legend(fontsize=9)

    axes[2].plot(epochs, history['lr'], color=colour)
    axes[2].set_xlabel('epoch')
    axes[2].set_ylabel('learning rate')
    axes[2].set_title('Learning rate')
    axes[2].set_yscale('log')

    fig.suptitle(f"{name} at {SIZE_256}: best epoch {best}, "
                 f"stopped at {stop}, val Dice {checkpoint['val_dice']:.4f}", y=1.04)
    fig.tight_layout()
    out_name = MODEL_FILES[name].replace('.pth', '_curves_256.png')
    fig.savefig(os.path.join(CURVES_DIR, out_name), bbox_inches='tight')
    plt.show()

In [ ]:
# all five on shared axes, so convergence speed and final level are comparable
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))

for name in MODEL_FILES:
    history = ckpt_256[name]['history']
    colour = COLOURS[name]
    epochs = np.arange(1, len(history['val_loss']) + 1)
    axes[0].plot(epochs, history['val_loss'], color=colour, label=name)
    axes[1].plot(epochs, history['val_dice'], color=colour, label=name)
    axes[2].plot(epochs, history['lr'], color=colour, label=name)

axes[0].set_xlabel('epoch')
axes[0].set_ylabel('validation loss')
axes[0].set_title('Validation loss')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('validation Dice')
axes[1].set_title('Validation Dice')
axes[1].legend(fontsize=9, loc='lower right')
axes[2].set_xlabel('epoch')
axes[2].set_ylabel('learning rate')
axes[2].set_title('Learning rate')
axes[2].set_yscale('log')

fig.suptitle(f"All five models at {SIZE_256}", y=1.03)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '03_training_curves_all_256.png'),
            bbox_inches='tight')
plt.show()

**How to read it and two traps.** Five models on shared axes: error going down, quality going up,
step size on the right.

**Trap one: the left panel cannot be compared across all five models.** Soft Erosion Hybrid is
measured on a different loss function, which is by construction 0.8 times the one the other four
use, plus a small extra term. It sits lowest for arithmetic reasons, not quality reasons. **Only the
middle panel, validation Dice, is comparable across all five.**

**Trap two: the two lowest lines on the right panel are not slower models.** What is recorded there
is the rate for the pretrained part of those two models; the rest of them trains at the same rate as
everything else.

Soft Erosion Hybrid, Hybrid U-Net and OCT-F Backbone converge to similar validation Dice, with
Modified OCT-F 2ch clearly below. The `ReduceLROnPlateau` steps are visible as drops in the
learning-rate panel, each one following a plateau in validation loss.

## 5. Generalisation gap at 256 x 256

Whether early stopping fired, and how far training Dice sits above validation Dice at the saved
epoch. A wide gap means the model fitted the 59 training images harder than the 15 it is judged on.

In [ ]:
# early stopping fired if training ended exactly patience epochs after the best,
# and the gap is read at the saved epoch so it describes the model actually kept
print(f"{'Model':<22} {'Best':>5} {'Stop':>5} {'Fired':>6} "
      f"{'Tr dice':>8} {'Val dice':>9} {'Gap':>7} {'Mins':>6}")
print("=" * 72)
for name in MODEL_FILES:
    checkpoint = ckpt_256[name]
    history = checkpoint['history']
    best = checkpoint['epoch']
    stop = checkpoint['config']['early_stopped_at']
    patience = checkpoint['config']['patience']

    if stop == best + patience:
        fired = 'yes'
    else:
        fired = 'no'

    train_dice = history['train_dice'][best - 1]
    val_dice = history['val_dice'][best - 1]
    minutes = sum(history['epoch_time']) / 60

    print(f"{name:<22} {best:>5} {stop:>5} {fired:>6} {train_dice:>8.4f} "
          f"{val_dice:>9.4f} {train_dice - val_dice:>7.4f} {minutes:>6.1f}")

In [ ]:
# ordered by the validation Dice recorded during training. these are
# training-loop values averaged over validation batches, not the reportable
# segmentation numbers, which notebook 04 computes per image
scores = {}
for name in MODEL_FILES:
    scores[name] = ckpt_256[name]['val_dice']

ranked = []
for name in MODEL_FILES:
    ranked.append((scores[name], name))
ranked.sort(reverse=True)

print(f"{'Model':<22} {'Best ep':>8} {'Val dice':>9}")
print("=" * 41)
for score, name in ranked:
    print(f"{name:<22} {ckpt_256[name]['epoch']:>8} {score:>9.4f}")

total = 0
for name in MODEL_FILES:
    total += sum(ckpt_256[name]['history']['epoch_time']) / 60
print(f"\nTotal training time at 256x256: {total:.1f} minutes")

Gaps range 0.0122 to 0.0956. Plain U-Net is narrowest and Modified OCT-F 2ch widest the first
has no pretrained weights to carry, the second adapts a pretrained encoder to 59 images. Ranked by
training-loop validation Dice: Soft Erosion 0.8612, Hybrid 0.8573, Plain 0.8395.

## 6. Protocol and convergence at native 496 x 768

The same five models at the acquisition geometry. Every setting is unchanged except the input size
and the two kernels that scale with it, so resolution is the only variable between this section
and Section 3.

In [ ]:
config = ckpt_native['Plain U-Net']['config']
print(f"Input size:      {SIZE_NATIVE}")
print(f"Batch size:      {config['batch_size']}")
print(f"Patience:        {config['patience']}")
print(f"PP kernel:       {config['pp_kernel']}")
print(f"Erosion kernel:  {config['erosion_kernel']}")
print("Split:           59 train / 15 validation, seed 40494109")

In [ ]:
# the epoch kept is the one with the lowest validation loss, not the last
print(f"{'Model':<22} {'Epochs':>7} {'Best':>6} {'Stop':>6} "
      f"{'Val loss':>10} {'Val dice':>10}")
print("=" * 64)
for name in MODEL_FILES:
    checkpoint = ckpt_native[name]
    n_epochs = len(checkpoint['history']['train_loss'])
    stop = checkpoint['config']['early_stopped_at']
    print(f"{name:<22} {n_epochs:>7} {checkpoint['epoch']:>6} {stop:>6} "
          f"{checkpoint['val_loss']:>10.4f} {checkpoint['val_dice']:>10.4f}")

All five terminated on the early-stopping criterion, between epochs 46 and 75 none reached the
ceiling. The PP and erosion kernels are 29 and 9 here against 15 and 5 at 256, scaled with the
vertical dimension.

## 7. Training curves at native 496 x 768

The same figures for the native run. Drawn with the same code and the same colours, so the two
sections can be read against each other directly.

In [ ]:
# one figure per model. the gap between the solid and dashed Dice lines is the
# generalisation signal, and the dotted line marks the epoch actually kept
for name in MODEL_FILES:
    checkpoint = ckpt_native[name]
    history = checkpoint['history']
    colour = COLOURS[name]
    best = checkpoint['epoch']
    stop = checkpoint['config']['early_stopped_at']
    epochs = np.arange(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

    axes[0].plot(epochs, history['train_loss'], color=colour, label='train')
    axes[0].plot(epochs, history['val_loss'], color=colour, linestyle='--',
                 label='validation')
    axes[0].axvline(best, color='0.4', linestyle=':', label=f"best {best}")
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('loss')
    axes[0].set_title('Loss')
    axes[0].legend(fontsize=9)

    axes[1].plot(epochs, history['train_dice'], color=colour, label='train')
    axes[1].plot(epochs, history['val_dice'], color=colour, linestyle='--',
                 label='validation')
    axes[1].axvline(best, color='0.4', linestyle=':')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('Dice')
    axes[1].set_title('Dice, higher is better')
    axes[1].legend(fontsize=9)

    axes[2].plot(epochs, history['lr'], color=colour)
    axes[2].set_xlabel('epoch')
    axes[2].set_ylabel('learning rate')
    axes[2].set_title('Learning rate')
    axes[2].set_yscale('log')

    fig.suptitle(f"{name} at {SIZE_NATIVE}: best epoch {best}, "
                 f"stopped at {stop}, val Dice {checkpoint['val_dice']:.4f}", y=1.04)
    fig.tight_layout()
    out_name = MODEL_FILES[name].replace('.pth', '_curves.png')
    fig.savefig(os.path.join(CURVES_DIR, out_name), bbox_inches='tight')
    plt.show()

In [ ]:
# all five on shared axes, so convergence speed and final level are comparable
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))

for name in MODEL_FILES:
    history = ckpt_native[name]['history']
    colour = COLOURS[name]
    epochs = np.arange(1, len(history['val_loss']) + 1)
    axes[0].plot(epochs, history['val_loss'], color=colour, label=name)
    axes[1].plot(epochs, history['val_dice'], color=colour, label=name)
    axes[2].plot(epochs, history['lr'], color=colour, label=name)

axes[0].set_xlabel('epoch')
axes[0].set_ylabel('validation loss')
axes[0].set_title('Validation loss')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('validation Dice')
axes[1].set_title('Validation Dice')
axes[1].legend(fontsize=9, loc='lower right')
axes[2].set_xlabel('epoch')
axes[2].set_ylabel('learning rate')
axes[2].set_title('Learning rate')
axes[2].set_yscale('log')

fig.suptitle(f"All five models at {SIZE_NATIVE}", y=1.03)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '03_training_curves_all.png'),
            bbox_inches='tight')
plt.show()

**How to read it and two traps.** Five models on shared axes: error down, quality up, step size
right.

**Trap one: the left panel is not a fair comparison.** Soft Erosion Hybrid optimises a different
loss, mechanically about 20% smaller than the one the other four are scored on. Comparing them there
is meaningless. Use the middle panel, validation Dice, which is the same measurement for every model.

**Trap two: the two low lines on the right are not slow training.** That panel records the rate of
the pretrained portion of the two OCT-F models; their new layers train at the usual rate.

Plain U-Net, Soft Erosion Hybrid and Hybrid U-Net converge near 0.84 validation Dice, with the
two pretrained models below them. Modified OCT-F 2ch separates from the rest earlier here than at
256.

## 8. Generalisation gap at native 496 x 768

The same measurement at native geometry. Comparing this table against Section 5 shows whether
resolution changes how hard each model fits its training set.

In [ ]:
# early stopping fired if training ended exactly patience epochs after the best,
# and the gap is read at the saved epoch so it describes the model actually kept
print(f"{'Model':<22} {'Best':>5} {'Stop':>5} {'Fired':>6} "
      f"{'Tr dice':>8} {'Val dice':>9} {'Gap':>7} {'Mins':>6}")
print("=" * 72)
for name in MODEL_FILES:
    checkpoint = ckpt_native[name]
    history = checkpoint['history']
    best = checkpoint['epoch']
    stop = checkpoint['config']['early_stopped_at']
    patience = checkpoint['config']['patience']

    if stop == best + patience:
        fired = 'yes'
    else:
        fired = 'no'

    train_dice = history['train_dice'][best - 1]
    val_dice = history['val_dice'][best - 1]
    minutes = sum(history['epoch_time']) / 60

    print(f"{name:<22} {best:>5} {stop:>5} {fired:>6} {train_dice:>8.4f} "
          f"{val_dice:>9.4f} {train_dice - val_dice:>7.4f} {minutes:>6.1f}")

In [ ]:
# ordered by the validation Dice recorded during training. these are
# training-loop values averaged over validation batches, not the reportable
# segmentation numbers, which notebook 04 computes per image
scores = {}
for name in MODEL_FILES:
    scores[name] = ckpt_native[name]['val_dice']

ranked = []
for name in MODEL_FILES:
    ranked.append((scores[name], name))
ranked.sort(reverse=True)

print(f"{'Model':<22} {'Best ep':>8} {'Val dice':>9}")
print("=" * 41)
for score, name in ranked:
    print(f"{name:<22} {ckpt_native[name]['epoch']:>8} {score:>9.4f}")

total = 0
for name in MODEL_FILES:
    total += sum(ckpt_native[name]['history']['epoch_time']) / 60
print(f"\nTotal training time at native 496x768: {total:.1f} minutes")

Modified OCT-F 2ch shows the widest gap at **0.1381**: train Dice 0.9289 against validation
0.7908. Hybrid U-Net is the only negative gap in either run at **−0.0010**, meaning it scored
fractionally higher on validation than on training at its saved epoch. Ranked: Plain 0.8465, Soft
Erosion 0.8430, Hybrid 0.8413.

## 9. Resolution comparison

The two runs side by side, on the training-loop validation Dice and the generalisation gap. This is
the only place the two are compared numerically; notebook 04 compares them on per-image results.

In [ ]:
print(f"{'Model':<22} {'256 dice':>9} {'Nat dice':>9} {'Diff':>8} "
      f"{'256 gap':>9} {'Nat gap':>9} {'256 ep':>7} {'Nat ep':>7}")
print("=" * 88)

for name in MODEL_FILES:
    small = ckpt_256[name]
    large = ckpt_native[name]

    small_best = small['epoch']
    large_best = large['epoch']
    small_gap = (small['history']['train_dice'][small_best - 1]
                 - small['history']['val_dice'][small_best - 1])
    large_gap = (large['history']['train_dice'][large_best - 1]
                 - large['history']['val_dice'][large_best - 1])
    difference = small['val_dice'] - large['val_dice']

    print(f"{name:<22} {small['val_dice']:>9.4f} {large['val_dice']:>9.4f} "
          f"{difference:>+8.4f} {small_gap:>9.4f} {large_gap:>9.4f} "
          f"{small_best:>7} {large_best:>7}")

Four of five score higher at 256 on this measure, Plain U-Net excepted. The two pretrained models
show the largest resolution difference, and also the widest gaps, consistent with a pretrained
encoder being more sensitive to how far the input geometry is distorted.

## 10. Convergence behaviour at both input sizes

Validation loss and validation Dice for all five models at both input sizes, on shared vertical
scales so the two runs can be compared by eye.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 8), sharey='row')

for name in MODEL_FILES:
    colour = COLOURS[name]
    small = ckpt_256[name]['history']
    large = ckpt_native[name]['history']
    small_epochs = np.arange(1, len(small['val_loss']) + 1)
    large_epochs = np.arange(1, len(large['val_loss']) + 1)

    axes[0][0].plot(small_epochs, small['val_loss'], color=colour, label=name)
    axes[0][1].plot(large_epochs, large['val_loss'], color=colour, label=name)
    axes[1][0].plot(small_epochs, small['val_dice'], color=colour, label=name)
    axes[1][1].plot(large_epochs, large['val_dice'], color=colour, label=name)

axes[0][0].set_title('Validation loss at 256 x 256')
axes[0][1].set_title('Validation loss at 496 x 768')
axes[1][0].set_title('Validation Dice at 256 x 256')
axes[1][1].set_title('Validation Dice at 496 x 768')
axes[1][0].legend(fontsize=9, loc='lower right')

for row in range(2):
    for column in range(2):
        axes[row][column].set_xlabel('epoch')

fig.suptitle('The same five models at both input sizes', y=1.01)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '03_resolution_curves.png'), bbox_inches='tight')
plt.show()

**How to read it.** The same five models drawn at two input sizes smaller on the left, full size on
the right with **the two panels in each row sharing a vertical scale**, which is what makes them
comparable by eye.

Compare left against right rather than line against line. The shapes and the point at which they
level off are similar, so neither run is unstable. What differs is how far apart the five lines
spread, and most of that spread is one model sitting below the rest.

The top row is error, so the loss caveat above applies to it as well.

Curve shape and convergence epoch are comparable at both input sizes, so neither run is unstable
relative to the other. The separation between models is wider at native, driven mainly by Modified
OCT-F 2ch.

## 11. Generalisation gap across both runs

The gaps from Sections 5 and 8 as one chart, which is easier to read across five models and two
input sizes than two separate tables.

In [ ]:
names = []
gaps_256 = []
gaps_native = []

for name in MODEL_FILES:
    small = ckpt_256[name]
    large = ckpt_native[name]
    small_best = small['epoch']
    large_best = large['epoch']

    names.append(name.replace(' ', '\n', 1))
    gaps_256.append(small['history']['train_dice'][small_best - 1]
                    - small['history']['val_dice'][small_best - 1])
    gaps_native.append(large['history']['train_dice'][large_best - 1]
                       - large['history']['val_dice'][large_best - 1])

x = np.arange(len(names))
width = 0.38

fig, ax = plt.subplots(figsize=(12, 4.4))
ax.bar(x - width / 2, gaps_256, width, label='256 x 256', color='#2F6FA8',
       edgecolor='white')
ax.bar(x + width / 2, gaps_native, width, label='496 x 768', color='#C0392B',
       edgecolor='white')
ax.axhline(0, color='0.4', linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylabel('train Dice − validation Dice')
ax.set_title('Generalisation gap at the saved epoch: wider means it fitted training harder')
ax.legend()

fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '03_generalisation_gap.png'), bbox_inches='tight')
plt.show()

**How to read it.** Bar height is the difference between how well a model does on the images it
learned from and the images it is judged on. **Taller means it fitted its training images harder
than it generalised.** Two bars per model, one for each input size.

The two models on the left were built from scratch and stay low. The two in the middle-right reuse a
pretrained encoder and are the tallest a strong pretrained model adapted to only 59 images fits
them much harder than it generalises. One bar dips below zero, which simply means that model scored
fractionally better on the judged images, because the training images are augmented and the judged
ones are not.

Modified OCT-F 2ch is widest at both input sizes. The only negative gap belongs to Hybrid U-Net at
native. The from-scratch U-Nets sit lower than the pretrained encoders throughout, which is the
clearest structural pattern in the two runs.

## 12. Output manifest

Every figure this notebook writes, for the figure catalogue in the supporting material.

In [ ]:
print("Figures saved:")
for name in MODEL_FILES:
    print(f"  outputs/training_curves/{MODEL_FILES[name].replace('.pth', '_curves_256.png')}")
for name in MODEL_FILES:
    print(f"  outputs/training_curves/{MODEL_FILES[name].replace('.pth', '_curves.png')}")
print("  outputs/images/03_training_curves_all_256.png")
print("  outputs/images/03_training_curves_all.png")
print("  outputs/images/03_resolution_curves.png")
print("  outputs/images/03_generalisation_gap.png")

---

## Summary of notebook 03

Ten runs reviewed from their stored histories: five models at **256 x 256** and the same five at
native **496 x 768**. Nothing was retrained.

**Every run stopped on its own criterion.** In all ten cases training ended exactly `patience`
epochs after the best epoch, between epochs 46 and 75, and none reached the 150-epoch ceiling so
every result is converged rather than truncated by a budget.

**The pairs are properly controlled.** Plain → Hybrid is **+288** parameters, OCT-F → Modified
OCT-F is **+384**, and Hybrid → Soft Erosion is **+0**. A difference within a pair is attributable
to the variable, not to capacity.

**The generalisation gap separates the two families.** The from-scratch U-Nets sit between −0.0010
and 0.0400; the pretrained encoders reach **0.0956** at 256 and **0.1381** at native. Modified
OCT-F 2ch is widest at both sizes a pretrained encoder on 59 training images fits them
considerably harder than it generalises.

Total training time is **3.1 minutes** at 256 and **7.2 minutes** at native.

**These are training-loop values**, averaged over validation batches during training. They drove
early stopping and are not the reportable segmentation numbers.